# 02 — Feature Engineering

Turn the raw team-game rows from notebook 01 into a model-ready feature matrix: one row per
game, with the Four Factors as rolling pre-game averages (no leakage), plus pace and rest as
separate challenger features. No modelling here.

**Input:** `data/raw/team_games_all.csv`
**Output:** `data/processed/team_games_features.csv`


## Setup and load

Locate the data folders and load the raw team-game rows (two rows per game).


In [1]:
from pathlib import Path

import pandas as pd

# Locate project folders.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "team_games_all.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load the raw team-game rows (two rows per game).
games = pd.read_csv(RAW_PATH, dtype={"GAME_ID": str}, parse_dates=["Date"])

## Step 1 — Four Factors per team-game

Compute the four factors on each team-game row. ORB% needs the opponent's defensive rebounds,
taken from the other team's row in the same `GAME_ID` (not the team's own DRB).


In [2]:
# Opponent DRB = the other team's DRB in the same game.
# Each GAME_ID has exactly two rows, so the game total minus own DRB leaves the opponent's.
games["opp_DRB"] = games.groupby("GAME_ID")["DRB"].transform("sum") - games["DRB"]

# Four Factors, computed per team-game row.
games["eFG"] = (games["FGM"] + 0.5 * games["FG3M"]) / games["FGA"]
games["TOV_pct"] = games["TOV"] / (games["FGA"] + 0.44 * games["FTA"] + games["TOV"])
games["ORB_pct"] = games["ORB"] / (games["ORB"] + games["opp_DRB"])
games["FTrate"] = games["FTM"] / games["FGA"]

FACTORS = ["eFG", "TOV_pct", "ORB_pct", "FTrate"]

## Step 2 — Challenger features

Pace (a per-game possessions estimate) and rest (days since the team's previous game). These
are kept separate from the four factors and are not part of the core four-factor importance
ranking later.


In [3]:
# Pace: per-game possessions estimate.
games["pace"] = games["FGA"] + 0.44 * games["FTA"] - games["ORB"] + games["TOV"]

# Rest: days since the team's previous game in the same season.
# Sorted so each team's games run in date order; the first game of a season has no prior
# game and stays NaN.
games = games.sort_values(["Team", "Season", "Date"])
games["rest"] = games.groupby(["Team", "Season"])["Date"].diff().dt.days

## Step 3 — Rolling pre-game averages

Replace each factor (and pace) with a 10-game rolling mean of the team's previous games, so a
game's features never use its own result. `shift(1)` drops the current game, grouping by
Team + Season stops any bleed across seasons, and averaging starts once a team has 5 games.
Rest stays a current-game value.


In [4]:
# Rolling pre-game averages for the four factors and pace.
ROLL_COLS = FACTORS + ["pace"]

for col in ROLL_COLS:
    games[col + "_roll"] = (
        games.groupby(["Team", "Season"])[col]
        .transform(lambda s: s.shift(1).rolling(10, min_periods=5).mean())
    )

## Step 4 — One row per game

Reshape so each game is a single row with the home and away values side by side, plus their
difference, for every factor and challenger. The `Home` flag assigns the sides.


In [5]:
# Map each rolled/rest column to its short feature name.
SIDE_MAP = {
    "eFG_roll": "eFG",
    "TOV_pct_roll": "TOV_pct",
    "ORB_pct_roll": "ORB_pct",
    "FTrate_roll": "FTrate",
    "pace_roll": "pace",
    "rest": "rest",
}

# Split each game into its home and away rows.
home = games[games["Home"] == 1]
away = games[games["Home"] == 0]

home_feats = home[["GAME_ID"] + list(SIDE_MAP)].rename(
    columns={col: "home_" + name for col, name in SIDE_MAP.items()}
)
away_feats = away[["GAME_ID"] + list(SIDE_MAP)].rename(
    columns={col: "away_" + name for col, name in SIDE_MAP.items()}
)

# Identifiers and target taken from the home row (y = home team win).
ids = home[["GAME_ID", "Date", "Season", "era", "W"]].rename(columns={"W": "y"})

matrix = ids.merge(home_feats, on="GAME_ID").merge(away_feats, on="GAME_ID")

# Difference columns: home minus away.
for name in SIDE_MAP.values():
    matrix["diff_" + name] = matrix["home_" + name] - matrix["away_" + name]

## Step 5 — Target and save

Order the columns (identifiers, the 12 four-factor features, then the pace and rest
challengers), drop early-season games whose rolling features never reached 5 prior games, and
save the feature matrix.


In [6]:
# Final column order: identifiers, the 12 four-factor features, then pace and rest challengers.
id_cols = ["GAME_ID", "Date", "Season", "era", "y"]

four_factor_cols = []
for name in FACTORS:
    four_factor_cols += [f"home_{name}", f"away_{name}", f"diff_{name}"]

pace_cols = ["home_pace", "away_pace", "diff_pace"]
rest_cols = ["home_rest", "away_rest", "diff_rest"]
feature_cols = four_factor_cols + pace_cols + rest_cols

matrix = matrix[id_cols + feature_cols].sort_values("Date").reset_index(drop=True)

# Drop early-season games whose rolling features never reached min_periods (NaN). Do not fill.
matrix = matrix.dropna(subset=feature_cols).reset_index(drop=True)

out_path = PROCESSED_DIR / "team_games_features.csv"
matrix.to_csv(out_path, index=False)
print(f"Saved {len(matrix)} games with {len(feature_cols)} features -> {out_path.name}")

Saved 11109 games with 18 features -> team_games_features.csv
